In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    python-dotenv \
    redis

In [0]:
dbutils.library.restartPython()

### 환경설정

In [0]:
import os
import sys
import json

os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()
print("[OK] vault 연결 완료")

### Redis 연결 + 키 정의

In [0]:
from redis.cluster import RedisCluster, ClusterNode

redis_host     = vault.get_secret("redis-host")
redis_password = vault.get_secret("redis-password")
redis_port     = int(vault.get_secret("redis-port"))

r = RedisCluster(
    startup_nodes=[ClusterNode(redis_host, redis_port)],
    password=redis_password,
    ssl=True,
    ssl_check_hostname=False,
    decode_responses=True,
    skip_full_coverage_check=True,
    socket_connect_timeout=10,
    socket_timeout=10,
)
r.ping()
print(f"[OK] Redis 연결 완료: {redis_host}:{redis_port}")

# ── 삭제 전 현재 상태 확인 ────────────────────────────────
all_rule_keys   = r.keys("gx_rules:*")
all_schema_keys = r.keys("gx_schema:*")

print(f"\n[삭제 전 상태]")
print(f"  규칙 키  : {all_rule_keys}")
print(f"  스키마 키: {all_schema_keys}")

for key in all_rule_keys:
    raw = r.get(key)
    if raw:
        rules = json.loads(raw)
        print(f"\n  [{key}]")
        print(f"    도메인  : {rules.get('domain')}")
        print(f"    버전    : {rules.get('version', '없음')}")
        print(f"    생성일  : {rules.get('generated_at')}")
        print(f"    규칙 수 : {len(rules.get('expectations', []))}개")

### 삭제 전 현재 상태 확인

In [0]:
# ── 삭제 실행 ─────────────────────────────────────────────
all_keys = r.keys("gx_rules:*") + r.keys("gx_schema:*")

for key in all_keys:
    r.delete(key)
    print(f"  [OK] 삭제: {key}")

print("\n[OK] 전체 Redis 캐시 삭제 완료")

### 삭제 실행

In [0]:
# ── 삭제 후 확인 ──────────────────────────────────────────
remaining_rules  = r.keys("gx_rules:*")
remaining_schema = r.keys("gx_schema:*")

print("[삭제 후 상태]")
print(f"  규칙 키  : {remaining_rules}")
print(f"  스키마 키: {remaining_schema}")

if not remaining_rules and not remaining_schema:
    print("\n→ 전부 삭제 완료")
    print("→ 01_rules_generator 실행 가능")
else:
    print("\n[WARN] 아직 남은 키가 있습니다")

### 삭제 후 확인